# 03 DuckDB SQL: public health questions on the analytic cohort

Answers the headline questions a health department or insurer would ask, in SQL, straight from
`data/processed/analytic_cohort.parquet`.

**Read every number here as crude and unweighted.** They describe the 50,819 NHANES adults in the
cohort, not the US population. Population estimates need the survey design (strata, PSU, weights)
and come from R in notebook 05 onward. Crude rates are also not age-adjusted, so a group that is
older will look worse partly because of age. Question 5 shows how big that effect is.

Death rates are **per 1,000 person-years**: deaths divided by total years of follow-up, which
handles people being followed for different lengths of time.

Each query result is saved to `outputs/sql_results/` for the dashboard.

In [1]:
!pip -q install duckdb
import duckdb, pandas as pd, pathlib

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = pathlib.Path("/content/drive/MyDrive/NHANES Cardiometabolic Mortality Project")
except ImportError:
    ROOT = pathlib.Path.cwd().parent

assert (ROOT / "notebooks").exists(), f"Project folder not found at {ROOT}"
SQL_DIR = ROOT / "outputs" / "sql_results"
SQL_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"CREATE VIEW cohort AS SELECT * FROM read_parquet('{ROOT / 'data' / 'processed' / 'analytic_cohort.parquet'}')")

def run(name, sql):
    df = con.sql(sql).df()
    df.to_csv(SQL_DIR / f"{name}.csv", index=False)
    return df

Mounted at /content/drive


## 1. Cohort at a glance

How many people, how many deaths, how long they were followed.

In [2]:
q01_overview = """
SELECT
    COUNT(*)                                                 AS participants,
    SUM(death)::INT                                          AS deaths,
    ROUND(SUM(time_months) / 12, 0)::INT                     AS person_years,
    ROUND(1000 * SUM(death) / (SUM(time_months) / 12), 2)    AS deaths_per_1000_py,
    ROUND(MEDIAN(time_months) / 12, 1)                       AS median_followup_years,
    ROUND(MAX(time_months) / 12, 1)                          AS max_followup_years
FROM cohort
"""
run("q01_overview", q01_overview)

,participants,deaths,person_years,deaths_per_1000_py,median_followup_years,max_followup_years
0,50819,8279,479971,17.25,9.0,20.8


## 2. Death rate by glycemic status

Does mortality climb from normal glucose to prediabetes to diabetes?

In [3]:
q02_glycemic = """
SELECT
    glycemic_status,
    COUNT(*)                                                 AS participants,
    SUM(death)::INT                                          AS deaths,
    ROUND(1000 * SUM(death) / (SUM(time_months) / 12), 2)    AS deaths_per_1000_py,
    ROUND(AVG(age), 1)                                       AS mean_age
FROM cohort
WHERE glycemic_status IS NOT NULL
GROUP BY glycemic_status
ORDER BY deaths_per_1000_py
"""
run("q02_glycemic", q02_glycemic)

,glycemic_status,participants,deaths,deaths_per_1000_py,mean_age
0,normal,24885,2808,10.76,43.9
1,prediabetes,15175,2555,19.42,54.9
2,diabetes,8575,2520,37.73,61.6


## 3. Cardiometabolic burden: how many conditions stack up

Counts diabetes, hypertension and obesity (BMI 30+) per person, 0 to 3. The question is whether
risk rises steadily with each extra condition, and what share of all deaths each level accounts for.

In [4]:
q03_burden = """
WITH scored AS (
    SELECT
        death,
        time_months,
        diabetes + hypertension + (bmi >= 30)::INT            AS n_conditions
    FROM cohort
    WHERE diabetes IS NOT NULL
      AND hypertension IS NOT NULL
      AND bmi IS NOT NULL
)
SELECT
    n_conditions::INT                                                     AS n_conditions,
    COUNT(*)                                                              AS participants,
    ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)                      AS pct_of_cohort,
    SUM(death)::INT                                                       AS deaths,
    ROUND(100 * SUM(death) / SUM(SUM(death)) OVER (), 1)                  AS pct_of_deaths,
    ROUND(1000 * SUM(death) / (SUM(time_months) / 12), 2)                 AS deaths_per_1000_py
FROM scored
GROUP BY n_conditions
ORDER BY n_conditions
"""
run("q03_burden", q03_burden)

,n_conditions,participants,pct_of_cohort,deaths,pct_of_deaths,deaths_per_1000_py
0,0,18540,39.9,1380,19.1,7.16
1,1,16053,34.6,2927,40.4,19.73
2,2,8544,18.4,2051,28.3,28.01
3,3,3280,7.1,880,12.2,34.31


## 4. Riskiest profiles, ranked

Every combination of diabetes, hypertension, obesity and CKD, ranked by death rate. Profiles
with fewer than 200 people are left out so a handful of deaths can't top the list.

In [5]:
q04_profiles = """
WITH profiles AS (
    SELECT
        CONCAT_WS(' + ',
            CASE WHEN diabetes = 1     THEN 'Diabetes'     END,
            CASE WHEN hypertension = 1 THEN 'Hypertension' END,
            CASE WHEN bmi >= 30        THEN 'Obesity'      END,
            CASE WHEN ckd = 1          THEN 'CKD'          END)            AS profile,
        death,
        time_months
    FROM cohort
    WHERE diabetes IS NOT NULL
      AND hypertension IS NOT NULL
      AND bmi IS NOT NULL
      AND ckd IS NOT NULL
)
SELECT
    RANK() OVER (ORDER BY 1000 * SUM(death) / (SUM(time_months) / 12) DESC)   AS risk_rank,
    COALESCE(NULLIF(profile, ''), 'None of the four')                         AS profile,
    COUNT(*)                                                                  AS participants,
    SUM(death)::INT                                                           AS deaths,
    ROUND(1000 * SUM(death) / (SUM(time_months) / 12), 2)                     AS deaths_per_1000_py
FROM profiles
GROUP BY profile
HAVING COUNT(*) >= 200
ORDER BY risk_rank
"""
run("q04_profiles", q04_profiles)

,risk_rank,profile,participants,deaths,deaths_per_1000_py
0,1,Diabetes + Hypertension + CKD,1092,573,78.71
1,2,Hypertension + CKD,2067,1057,70.82
2,3,Diabetes + CKD,320,126,52.31
3,4,Diabetes + Hypertension + Obesity + CKD,1469,554,51.35
4,5,Hypertension + Obesity + CKD,1173,407,42.33
5,6,Diabetes + Obesity + CKD,330,96,38.82
6,7,Diabetes + Hypertension,1187,315,32.85
7,8,Hypertension,5197,1179,24.00
8,9,CKD,1292,288,23.90
9,10,Diabetes + Hypertension + Obesity,1789,318,21.63


## 5. Age and sex: the confounders to keep in mind

Death rates by age group and sex. The gap between age groups is far larger than any gap in
questions 2 to 4, which is why every model in R adjusts for age.

In [6]:
q05_age_sex = """
SELECT
    age_group,
    sex_lbl                                                  AS sex,
    COUNT(*)                                                 AS participants,
    SUM(death)::INT                                          AS deaths,
    ROUND(1000 * SUM(death) / (SUM(time_months) / 12), 2)    AS deaths_per_1000_py
FROM cohort
GROUP BY age_group, sex_lbl
ORDER BY age_group, sex
"""
run("q05_age_sex", q05_age_sex)

,age_group,sex,participants,deaths,deaths_per_1000_py
0,20-39,female,8155,107,1.26
1,20-39,male,8364,199,2.26
2,40-59,female,8509,521,6.16
3,40-59,male,7996,752,9.46
4,60-79,female,7039,1775,28.12
5,60-79,male,7059,2402,40.97
6,80+,female,1964,1284,105.26
7,80+,male,1733,1239,134.47


## 6. Cause of death by glycemic status

Among people who died, what share died of CVD, cancer, or other causes? Diabetes is expected to
shift deaths toward CVD.

In [7]:
q06_cause_mix = """
SELECT
    glycemic_status,
    COUNT(*)                                                               AS deaths,
    ROUND(100 * AVG((cause = 'cvd')::INT), 1)                              AS pct_cvd,
    ROUND(100 * AVG((cause = 'cancer')::INT), 1)                           AS pct_cancer,
    ROUND(100 * AVG((cause = 'other')::INT), 1)                            AS pct_other,
    ROUND(100 * AVG((cause = 'unknown')::INT), 1)                          AS pct_unknown
FROM cohort
WHERE death = 1
  AND glycemic_status IS NOT NULL
GROUP BY glycemic_status
ORDER BY pct_cvd DESC
"""
run("q06_cause_mix", q06_cause_mix)

,glycemic_status,deaths,pct_cvd,pct_cancer,pct_other,pct_unknown
0,diabetes,2520,33.9,17.1,49.0,0.0
1,prediabetes,2555,31.3,25.4,43.4,0.0
2,normal,2808,30.0,22.3,47.7,0.0


## 7. Prevalence trend across survey cycles, 1999 to 2018

Crude share of the sample with each condition, cycle by cycle.

In [8]:
q07_trends = """
SELECT
    cycle,
    COUNT(*)                                                 AS participants,
    ROUND(100 * AVG(diabetes), 1)                            AS pct_diabetes,
    ROUND(100 * AVG(prediabetes), 1)                         AS pct_prediabetes,
    ROUND(100 * AVG(hypertension), 1)                        AS pct_hypertension,
    ROUND(100 * AVG((bmi >= 30)::INT), 1)                    AS pct_obese,
    ROUND(100 * AVG(ckd), 1)                                 AS pct_ckd,
    ROUND(AVG(hba1c), 2)                                     AS mean_hba1c
FROM cohort
GROUP BY cycle
ORDER BY cycle
"""
run("q07_trends", q07_trends)

,cycle,participants,pct_diabetes,pct_prediabetes,pct_hypertension,pct_obese,pct_ckd,mean_hba1c
0,1999-2000,4184,14.5,22.6,38.7,32.2,19.1,5.62
1,2001-2002,4726,13.9,25.0,37.0,30.3,18.4,5.63
2,2003-2004,4513,15.0,23.9,40.9,32.8,19.2,5.64
3,2005-2006,4446,15.0,27.6,36.8,35.2,18.3,5.61
4,2007-2008,5644,19.0,35.2,39.9,36.6,18.8,5.75
5,2009-2010,5977,17.5,34.7,37.7,38.1,16.0,5.76
6,2011-2012,5251,18.6,33.5,37.3,35.9,17.5,5.80
7,2013-2014,5513,17.7,32.0,37.2,37.8,17.3,5.75
8,2015-2016,5389,20.9,35.3,38.0,40.5,17.1,5.86
9,2017-2018,5176,22.6,37.3,43.4,41.9,19.5,5.88


## 8. Hypertension care cascade

Among people with hypertension: how many know about it, how many are treated, how many are
controlled (on medication and below 140/90). Each step of the cascade is a gap a health system can act on.

In [9]:
q08_htn_cascade = """
SELECT
    cycle,
    COUNT(*)                                                                         AS hypertensive,
    ROUND(100 * AVG(COALESCE(htn_told, 0)), 1)                                       AS pct_aware,
    ROUND(100 * AVG(COALESCE(htn_med, 0)), 1)                                        AS pct_treated,
    ROUND(100 * AVG((COALESCE(htn_med, 0) = 1 AND sbp < 140 AND dbp < 90)::INT), 1)  AS pct_controlled
FROM cohort
WHERE hypertension = 1
GROUP BY cycle
ORDER BY cycle
"""
run("q08_htn_cascade", q08_htn_cascade)

,cycle,hypertensive,pct_aware,pct_treated,pct_controlled
0,1999-2000,1577,69.4,60.0,26.7
1,2001-2002,1681,72.0,62.6,32.0
2,2003-2004,1753,77.0,68.2,37.2
3,2005-2006,1571,78.9,70.3,42.1
4,2007-2008,2173,80.3,73.6,46.3
5,2009-2010,2180,83.5,77.3,50.3
6,2011-2012,1892,83.3,76.1,49.3
7,2013-2014,2008,85.1,76.6,51.8
8,2015-2016,2002,82.7,74.9,46.2
9,2017-2018,2167,79.4,72.1,41.5


## 9. Diabetes: undiagnosed and poorly controlled

Among people with diabetes: what share has never been told (found only by the lab test), and
among the diagnosed, what share has HbA1c 9% or higher (poor control).

In [10]:
q09_diabetes_care = """
SELECT
    cycle,
    COUNT(*)                                                                         AS with_diabetes,
    ROUND(100 * AVG((COALESCE(dm_told, 0) = 0)::INT), 1)                             AS pct_undiagnosed,
    ROUND(100 * AVG(CASE WHEN dm_told = 1 THEN (hba1c >= 9)::INT END), 1)            AS pct_poor_control_diagnosed
FROM cohort
WHERE diabetes = 1
GROUP BY cycle
ORDER BY cycle
"""
run("q09_diabetes_care", q09_diabetes_care)

,cycle,with_diabetes,pct_undiagnosed,pct_poor_control_diagnosed
0,1999-2000,577,23.7,22.3
1,2001-2002,626,26.8,16.8
2,2003-2004,649,21.9,14.6
3,2005-2006,634,24.1,16.6
4,2007-2008,1016,28.9,14.6
5,2009-2010,1001,28.8,14.5
6,2011-2012,929,27.3,17.5
7,2013-2014,945,25.4,16.2
8,2015-2016,1075,25.1,17.6
9,2017-2018,1123,25.9,15.5


## 10. Kidney disease: KDIGO heat map

Death rate for every combination of eGFR stage (G1 to G5) and albuminuria stage (A1 to A3), the
grid nephrologists use to grade CKD risk. Cells with fewer than 30 people are blanked.

In [11]:
q10_kdigo = """
WITH staged AS (
    SELECT
        CASE WHEN egfr >= 90 THEN 'G1'
             WHEN egfr >= 60 THEN 'G2'
             WHEN egfr >= 45 THEN 'G3a'
             WHEN egfr >= 30 THEN 'G3b'
             WHEN egfr >= 15 THEN 'G4'
             ELSE 'G5' END                                   AS gfr_stage,
        CASE WHEN acr < 30   THEN 'A1'
             WHEN acr < 300  THEN 'A2'
             ELSE 'A3' END                                   AS alb_stage,
        death,
        time_months
    FROM cohort
    WHERE egfr IS NOT NULL
      AND acr IS NOT NULL
)
SELECT
    gfr_stage,
    alb_stage,
    COUNT(*)                                                                         AS participants,
    SUM(death)::INT                                                                  AS deaths,
    CASE WHEN COUNT(*) >= 30
         THEN ROUND(1000 * SUM(death) / (SUM(time_months) / 12), 2) END              AS deaths_per_1000_py
FROM staged
GROUP BY gfr_stage, alb_stage
ORDER BY gfr_stage, alb_stage
"""
run("q10_kdigo", q10_kdigo).pivot(index="gfr_stage", columns="alb_stage", values="deaths_per_1000_py")

alb_stage,A1,A2,A3
gfr_stage,,,
G1,5.87,16.74,27.38
G2,20.41,54.57,55.43
G3a,51.94,93.52,103.36
G3b,73.58,110.03,115.67
G4,109.02,122.41,153.38
G5,NaN,NaN,107.26


## 11. Life table in SQL: survival by glycemic status

An actuarial life table built with window functions: for each year of follow-up, how many were
still at risk, how many died, and the cumulative share surviving. People who leave the study
without dying (censored) count as at risk for half the year, the standard actuarial convention.
This is a preview of the Kaplan-Meier curves in R.

In [12]:
q11_life_table = """
WITH yearly AS (
    SELECT
        glycemic_status,
        FLOOR(time_months / 12)::INT                                           AS fu_year,
        COUNT(*)                                                               AS n_exit,
        SUM(death)                                                             AS n_death
    FROM cohort
    WHERE glycemic_status IS NOT NULL
    GROUP BY glycemic_status, fu_year
),
at_risk AS (
    SELECT
        glycemic_status,
        fu_year,
        n_death,
        n_exit - n_death                                                       AS n_censored,
        SUM(n_exit) OVER (PARTITION BY glycemic_status ORDER BY fu_year DESC)  AS n_at_risk
    FROM yearly
)
SELECT
    glycemic_status,
    fu_year,
    n_at_risk::INT                                                             AS n_at_risk,
    n_death::INT                                                               AS n_death,
    ROUND(100 * EXP(SUM(LN(GREATEST(1 - n_death / (n_at_risk - n_censored / 2), 1e-9)))
          OVER (PARTITION BY glycemic_status ORDER BY fu_year)), 1)            AS pct_surviving
FROM at_risk
ORDER BY glycemic_status, fu_year
"""
life = run("q11_life_table", q11_life_table)
life.pivot(index="fu_year", columns="glycemic_status", values="pct_surviving")

glycemic_status,diabetes,normal,prediabetes
fu_year,,,
0,97.7,99.4,99.1
1,95.1,98.7,97.7
2,92.0,97.9,96.1
3,89.1,97.0,94.6
4,85.9,95.9,93.1
5,82.4,94.8,91.2
6,78.9,93.8,89.2
7,75.8,92.7,87.1
8,72.5,91.7,85.2


## Saved outputs

All eleven result tables are in `outputs/sql_results/` as `q01_overview.csv` through
`q11_life_table.csv`.

**Next: notebook 04, EDA.** Distributions, correlations and plots of the same cohort, still crude,
to check shapes and relationships before the R models.

In [13]:
sorted(p.name for p in SQL_DIR.glob("q*.csv"))

['q01_overview.csv',
 'q02_glycemic.csv',
 'q03_burden.csv',
 'q04_profiles.csv',
 'q05_age_sex.csv',
 'q06_cause_mix.csv',
 'q07_trends.csv',
 'q08_htn_cascade.csv',
 'q09_diabetes_care.csv',
 'q10_kdigo.csv',
 'q11_life_table.csv']